# 📱 IG Manager Pro — بناء APK تلقائياً

**تعليمات:**
1. اضغط **Runtime → Run all** من القائمة أعلاه
2. انتظر 30-40 دقيقة (البناء الأول يُحمِّل NDK)
3. بعد الانتهاء، سيظهر رابط تحميل APK في آخر خلية

> ⚠️ تأكد أن Colab يعمل بـ **Python 3.10** (Runtime → Change runtime type)


In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 1: تثبيت المتطلبات الأساسية
# ═══════════════════════════════════════════════════════════
print('⏳ Installing system dependencies...')
import subprocess
cmds = [
    'apt-get update -qq',
    'apt-get install -y -qq python3-pip git zip unzip openjdk-17-jdk '
    'autoconf libtool pkg-config zlib1g-dev libncurses5-dev '
    'libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev',
]
for cmd in cmds:
    subprocess.run(cmd, shell=True, check=True, capture_output=True)

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
print('✅ System dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 2: تثبيت Python packages
# ═══════════════════════════════════════════════════════════
print('⏳ Installing Python packages...')
!pip install -q buildozer cython==0.29.36 kivy==2.3.0 kivymd==1.2.0
print('✅ Python packages installed')

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 3: إنشاء ملفات المشروع
# ═══════════════════════════════════════════════════════════
import os
os.makedirs('/content/igapp/screens', exist_ok=True)
os.makedirs('/content/igapp/core',    exist_ok=True)
os.makedirs('/content/igapp/models',  exist_ok=True)
os.makedirs('/content/igapp/assets',  exist_ok=True)

# ── assets/theme.py ─────────────────────────────────────────
open('/content/igapp/assets/__init__.py','w').close()
with open('/content/igapp/assets/theme.py','w') as f:
    f.write('''
PRIMARY_PURPLE = "#a855f7"
PRIMARY_CYAN   = "#06b6d4"
BG_DARK        = "#0a0a0f"
BG_CARD        = "#12121a"
BG_CARD2       = "#1a1a2e"
TEXT_PRIMARY   = "#f1f5f9"
TEXT_SECONDARY = "#94a3b8"
TEXT_MUTED     = "#475569"
STATUS_ACTIVE    = "#22c55e"
STATUS_DISABLED  = "#ef4444"
STATUS_SHADOW    = "#f97316"
STATUS_PRIVATE   = "#a855f7"
STATUS_VERIFIED  = "#06b6d4"
STATUS_UNKNOWN   = "#94a3b8"
''')

# ── models/database.py ─────────────────────────────────────
open('/content/igapp/models/__init__.py','w').close()
with open('/content/igapp/models/database.py','w') as f:
    f.write('''
import sqlite3, os, json
from datetime import datetime

def get_db_path():
    try:
        from android.storage import app_storage_path
        base = app_storage_path()
    except ImportError:
        base = os.path.join(os.path.expanduser("~"), ".igmanager")
    os.makedirs(base, exist_ok=True)
    return os.path.join(base, "igmanager.db")

class Database:
    def __init__(self):
        self.db_path = get_db_path()
        self._init_db()

    def _connect(self):
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_db(self):
        conn = self._connect()
        conn.execute("""
            CREATE TABLE IF NOT EXISTS scans (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL,
                status TEXT DEFAULT \'unknown\',
                followers INTEGER DEFAULT 0,
                following INTEGER DEFAULT 0,
                posts INTEGER DEFAULT 0,
                is_private INTEGER DEFAULT 0,
                is_verified INTEGER DEFAULT 0,
                shadowban INTEGER DEFAULT 0,
                full_name TEXT DEFAULT \'\',
                bio TEXT DEFAULT \'\',
                scanned_at TEXT DEFAULT (datetime(\'now\')),
                raw_data TEXT DEFAULT \'{}\'
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS appeals (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT, full_name TEXT, email TEXT,
                year_created TEXT, country TEXT,
                issue_type TEXT, issue_detail TEXT,
                appeal_ar TEXT, appeal_en TEXT,
                created_at TEXT DEFAULT (datetime(\'now\'))
            )
        """)
        conn.commit(); conn.close()

    def save_scan(self, data):
        conn = self._connect()
        conn.execute("""
            INSERT INTO scans
            (username,status,followers,following,posts,is_private,is_verified,shadowban,full_name,bio,scanned_at,raw_data)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            data.get("username",""), data.get("status","unknown"),
            data.get("followers",0), data.get("following",0),
            data.get("posts",0),
            1 if data.get("is_private") else 0,
            1 if data.get("is_verified") else 0,
            1 if data.get("shadowban") else 0,
            data.get("full_name",""), data.get("bio",""),
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            json.dumps({}),
        ))
        conn.commit(); conn.close()

    def get_all_scans(self, status_filter=None):
        conn = self._connect()
        if status_filter and status_filter != "all":
            rows = conn.execute("SELECT * FROM scans WHERE status=? ORDER BY scanned_at DESC",(status_filter,)).fetchall()
        else:
            rows = conn.execute("SELECT * FROM scans ORDER BY scanned_at DESC").fetchall()
        conn.close()
        return [dict(r) for r in rows]

    def get_recent_scans(self, limit=10):
        conn = self._connect()
        rows = conn.execute("SELECT * FROM scans ORDER BY scanned_at DESC LIMIT ?",(limit,)).fetchall()
        conn.close()
        return [dict(r) for r in rows]

    def get_stats(self):
        conn = self._connect()
        total = conn.execute("SELECT COUNT(*) FROM scans").fetchone()[0]
        by_s = {r[0]:r[1] for r in conn.execute("SELECT status,COUNT(*) FROM scans GROUP BY status").fetchall()}
        conn.close()
        return {"total":total,"active":by_s.get("active",0),
                "disabled":by_s.get("disabled",0),"shadowban":by_s.get("shadowban",0),
                "private":by_s.get("private",0),"unknown":by_s.get("unknown",0)}

    def clear_all_scans(self):
        conn = self._connect()
        conn.execute("DELETE FROM scans")
        conn.commit(); conn.close()

    def save_appeal(self, data):
        conn = self._connect()
        conn.execute("""
            INSERT INTO appeals
            (username,full_name,email,year_created,country,issue_type,issue_detail,appeal_ar,appeal_en,created_at)
            VALUES (?,?,?,?,?,?,?,?,?,?)
        """, (
            data.get("username",""),data.get("full_name",""),data.get("email",""),
            data.get("year_created",""),data.get("country",""),
            data.get("issue_type",""),data.get("issue_detail",""),
            data.get("appeal_ar",""),data.get("appeal_en",""),
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        ))
        conn.commit(); conn.close()
''')

# ── core/checker.py ─────────────────────────────────────────
open('/content/igapp/core/__init__.py','w').close()
with open('/content/igapp/core/checker.py','w') as f:
    f.write('''
import requests, json, time, re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 11; Samsung Galaxy S21) AppleWebKit/537.36",
    "Accept-Language": "en-US,en;q=0.5",
}
TIMEOUT = 15
_SESSION = None

def _s():
    global _SESSION
    if not _SESSION:
        _SESSION = requests.Session()
        _SESSION.headers.update(HEADERS)
    return _SESSION

def check_account(username):
    username = username.strip().lstrip("@").lower()
    result = {"username":username,"status":"unknown","followers":0,"following":0,
              "posts":0,"is_private":False,"is_verified":False,"shadowban":False,
              "full_name":"","bio":"","error":None}
    if not username:
        result["error"] = "Invalid username"; return result
    try:
        resp = _s().get(f"https://www.instagram.com/{username}/", timeout=TIMEOUT)
        if resp.status_code == 404:
            result["status"] = "disabled"; return result
        html = resp.text
        if "Sorry, this page" in html or "Page Not Found" in html:
            result["status"] = "disabled"; return result
        if "This account is private" in html:
            result["status"] = "private"; result["is_private"] = True; return result
        match = re.search(r\'"edge_followed_by":\{"count":(\\d+)\}\', html)
        if match:
            result["followers"] = int(match.group(1))
            result["status"] = "active" if result["followers"] > 0 else "shadowban"
        else:
            result["status"] = "active" if username in html else "unknown"
        m2 = re.search(r\'"full_name":"([^"]+)"\', html)
        if m2: result["full_name"] = m2.group(1)
        result["is_verified"] = \'"is_verified":true\' in html
    except Exception as e:
        result["error"] = str(e)
    return result

def check_accounts_bulk(usernames, progress_cb=None, stop_flag=None):
    results = []
    total = len(usernames)
    for idx, u in enumerate(usernames):
        if stop_flag and stop_flag(): break
        r = check_account(u)
        results.append(r)
        if progress_cb: progress_cb(idx+1, total, r)
        time.sleep(1.2)
    return results
''')

# ── core/appeals.py ────────────────────────────────────────
with open('/content/igapp/core/appeals.py','w') as f:
    f.write('''
from datetime import datetime

ISSUE_TYPES = {
    "disabled":      ("حساب معطل",       "Disabled Account"),
    "shadowban":     ("شادوبان",          "Shadowban"),
    "hacked":        ("حساب مخترق",       "Hacked Account"),
    "impersonation": ("انتحال هوية",      "Impersonation"),
    "copyright":     ("حقوق نشر",        "Copyright"),
    "other":         ("مشكلة أخرى",       "Other Issue"),
}
INSTAGRAM_SUPPORT_URL = "https://help.instagram.com/contact/1652567094844914"

def generate_appeal(data):
    u  = data.get("username","").strip().lstrip("@")
    fn = data.get("full_name","")
    em = data.get("email","")
    yr = data.get("year_created","")
    co = data.get("country","")
    it = data.get("issue_type","other")
    id_ = data.get("issue_detail","")
    iar, ien = ISSUE_TYPES.get(it, ISSUE_TYPES["other"])
    now = datetime.now().strftime("%Y-%m-%d")

    ar = f"""طلب استئناف رسمي – إنستغرام\nالتاريخ: {now}\n\nإلى فريق دعم إنستغرام،\n\nأنا {fn}، صاحب الحساب @{u}\nالبريد: {em} | السنة: {yr} | البلد: {co}\nالمشكلة: {iar}\n\nالتفاصيل:\n{id_}\n\nأطلب مراجعة الإجراء وإعادة الحساب لوضعه الطبيعي.\n\nمع التقدير،\n{fn}\n"""
    en = f"""Official Appeal – Instagram\nDate: {now}\n\nDear Instagram Support Team,\n\nI am {fn}, owner of account @{u}\nEmail: {em} | Year: {yr} | Country: {co}\nIssue: {ien}\n\nDetails:\n{id_}\n\nI request an urgent review and full restoration of my account.\n\nRespectfully,\n{fn}\n"""
    return {"appeal_ar": ar, "appeal_en": en}
''')

print('✅ Project files created')

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 4: إنشاء شاشات التطبيق
# ═══════════════════════════════════════════════════════════
open('/content/igapp/screens/__init__.py','w').close()

# ── screens/dashboard.py ────────────────────────────────────
with open('/content/igapp/screens/dashboard.py','w') as f:
    f.write('''
from kivy.lang import Builder
from kivymd.uix.screen import MDScreen
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.card import MDCard
from kivymd.uix.label import MDLabel
from kivy.metrics import dp
from kivy.utils import get_color_from_hex
from models.database import Database

Builder.load_string(\'\'\'<DashboardScreen>:
    name: "dashboard"
    MDBoxLayout:
        orientation: "vertical"
        md_bg_color: app.hex_rgba("#0a0a0f")
        MDBoxLayout:
            size_hint_y: None
            height: "60dp"
            padding: "16dp"
            md_bg_color: app.hex_rgba("#12121a")
            MDLabel:
                text: "[color=#a855f7]IG Manager Pro[/color]"
                markup: True
                font_style: "H6"
                bold: True
        ScrollView:
            MDBoxLayout:
                id: body
                orientation: "vertical"
                padding: "16dp"
                spacing: "12dp"
                size_hint_y: None
                height: self.minimum_height
                MDGridLayout:
                    id: stats_grid
                    cols: 2
                    spacing: "10dp"
                    size_hint_y: None
                    height: self.minimum_height
                MDCard:
                    id: recent_card
                    orientation: "vertical"
                    padding: "16dp"
                    spacing: "8dp"
                    radius: [14,]
                    md_bg_color: app.hex_rgba("#12121a")
                    size_hint_y: None
                    height: self.minimum_height
                    MDLabel:
                        text: "[color=#94a3b8]Recent Scans[/color]"
                        markup: True
                        font_style: "Subtitle1"
                        bold: True
                        size_hint_y: None
                        height: self.texture_size[1]
                    MDBoxLayout:
                        id: recent_box
                        orientation: "vertical"
                        spacing: "6dp"
                        size_hint_y: None
                        height: self.minimum_height
\'\'\' )

STAT_CFG = [
    ("Active",    "active",    "#22c55e"),
    ("Disabled",  "disabled",  "#ef4444"),
    ("Shadowban", "shadowban", "#f97316"),
    ("Private",   "private",   "#a855f7"),
]

class DashboardScreen(MDScreen):
    def on_enter(self): self.refresh()
    def refresh(self):
        db = Database()
        stats   = db.get_stats()
        recents = db.get_recent_scans(10)
        grid = self.ids.stats_grid
        grid.clear_widgets()
        for label, key, color in STAT_CFG:
            card = MDCard(orientation="vertical", padding="14dp",
                          radius=[14,], md_bg_color=get_color_from_hex("#12121a"),
                          size_hint_y=None, height=dp(100), elevation=3)
            card.add_widget(MDLabel(text=label, font_style="Caption",
                                    theme_text_color="Custom",
                                    text_color=get_color_from_hex("#94a3b8"),
                                    size_hint_y=None, height=dp(20)))
            card.add_widget(MDLabel(text=str(stats.get(key,0)), font_style="H4", bold=True,
                                    theme_text_color="Custom",
                                    text_color=get_color_from_hex(color)))
            grid.add_widget(card)
        box = self.ids.recent_box
        box.clear_widgets()
        STATUS_COLORS = {"active":"#22c55e","disabled":"#ef4444","shadowban":"#f97316",
                         "private":"#a855f7","unknown":"#94a3b8"}
        for s in recents:
            c = get_color_from_hex(STATUS_COLORS.get(s.get("status","unknown"),"#94a3b8"))
            row = MDBoxLayout(orientation="horizontal", size_hint_y=None, height=dp(32))
            row.add_widget(MDLabel(text=f"@{s.get(\'username\',\'\')}",
                                   theme_text_color="Custom", text_color=c,
                                   font_style="Body2", bold=True, size_hint_x=0.6))
            row.add_widget(MDLabel(text=f"{s.get(\'followers\',0):,} followers",
                                   theme_text_color="Custom",
                                   text_color=get_color_from_hex("#475569"),
                                   font_style="Caption", halign="right", size_hint_x=0.4))
            box.add_widget(row)
        if not recents:
            box.add_widget(MDLabel(text="No scans yet.", font_style="Body2",
                                   theme_text_color="Custom",
                                   text_color=get_color_from_hex("#475569"),
                                   size_hint_y=None, height=dp(36)))
''')

# ── screens/scanner.py ───────────────────────────────────────
with open('/content/igapp/screens/scanner.py','w') as f:
    f.write('''
import threading, os
from kivy.lang import Builder
from kivy.clock import Clock
from kivy.metrics import dp
from kivy.utils import get_color_from_hex
from kivymd.uix.screen import MDScreen
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.label import MDLabel
from models.database import Database
from core.checker import check_account, check_accounts_bulk

Builder.load_string(\'\'\'<ScannerScreen>:
    name: "scanner"
    MDBoxLayout:
        orientation: "vertical"
        md_bg_color: app.hex_rgba("#0a0a0f")
        MDBoxLayout:
            size_hint_y: None
            height: "60dp"
            padding: "16dp"
            md_bg_color: app.hex_rgba("#12121a")
            MDLabel:
                text: "[color=#06b6d4]Scanner[/color]"
                markup: True
                font_style: "H6"
                bold: True
        ScrollView:
            MDBoxLayout:
                orientation: "vertical"
                padding: "16dp"
                spacing: "14dp"
                size_hint_y: None
                height: self.minimum_height
                MDCard:
                    orientation: "vertical"
                    padding: "16dp"
                    spacing: "10dp"
                    radius: [14,]
                    md_bg_color: app.hex_rgba("#12121a")
                    size_hint_y: None
                    height: self.minimum_height
                    elevation: 4
                    MDLabel:
                        text: "[color=#06b6d4]Single Account Check[/color]"
                        markup: True
                        font_style: "Subtitle1"
                        bold: True
                        size_hint_y: None
                        height: self.texture_size[1]
                    MDTextField:
                        id: uname
                        hint_text: "@username"
                        mode: "rectangle"
                        size_hint_y: None
                        height: "48dp"
                    MDRaisedButton:
                        text: "CHECK ACCOUNT"
                        md_bg_color: app.hex_rgba("#a855f7")
                        size_hint_x: 1
                        on_release: root.single_scan()
                    MDBoxLayout:
                        id: result_box
                        orientation: "vertical"
                        spacing: "4dp"
                        size_hint_y: None
                        height: self.minimum_height
                MDCard:
                    orientation: "vertical"
                    padding: "16dp"
                    spacing: "10dp"
                    radius: [14,]
                    md_bg_color: app.hex_rgba("#12121a")
                    size_hint_y: None
                    height: self.minimum_height
                    elevation: 4
                    MDLabel:
                        text: "[color=#a855f7]Bulk Scan (.txt file)[/color]"
                        markup: True
                        font_style: "Subtitle1"
                        bold: True
                        size_hint_y: None
                        height: self.texture_size[1]
                    MDTextField:
                        id: fpath
                        hint_text: "/sdcard/Download/accounts.txt"
                        mode: "rectangle"
                        size_hint_y: None
                        height: "48dp"
                    MDRaisedButton:
                        id: bulk_btn
                        text: "START BULK SCAN"
                        md_bg_color: app.hex_rgba("#06b6d4")
                        size_hint_x: 1
                        on_release: root.bulk_scan()
                    MDProgressBar:
                        id: pbar
                        value: 0
                        color: app.hex_rgba("#a855f7")
                        size_hint_y: None
                        height: "6dp"
                    MDLabel:
                        id: plabel
                        text: "Idle"
                        theme_text_color: "Custom"
                        text_color: app.hex_rgba("#475569")
                        font_style: "Caption"
                        size_hint_y: None
                        height: "18dp"
                    ScrollView:
                        size_hint_y: None
                        height: "180dp"
                        MDLabel:
                            id: blog
                            text: ""
                            theme_text_color: "Custom"
                            text_color: app.hex_rgba("#94a3b8")
                            font_style: "Body2"
                            size_hint_y: None
                            height: self.texture_size[1]
                            text_size: self.width, None
\'\'\' )

SC = {"active":("OK","#22c55e"),"disabled":("OFF","#ef4444"),
      "shadowban":("SB","#f97316"),"private":("PRV","#a855f7"),"unknown":("?","#94a3b8")}

class ScannerScreen(MDScreen):
    _stop = False; _scanning = False
    def single_scan(self):
        u = self.ids.uname.text.strip().lstrip("@")
        if not u: return
        threading.Thread(target=self._do_single, args=(u,), daemon=True).start()
    def _do_single(self, u):
        r = check_account(u)
        if r.get("status") != "unknown": Database().save_scan(r)
        Clock.schedule_once(lambda dt: self._show(r))
    def _show(self, r):
        box = self.ids.result_box; box.clear_widgets()
        status = r.get("status","unknown")
        _, color = SC.get(status, ("?","#94a3b8"))
        c = get_color_from_hex(color)
        box.add_widget(MDLabel(text=f"@{r[\'username\']} — {status.upper()}",
                               theme_text_color="Custom", text_color=c,
                               font_style="Subtitle1", bold=True,
                               size_hint_y=None, height=dp(32)))
        for lbl, val in [("Followers", f"{r[\'followers\']:,}"),
                         ("Following", f"{r[\'following\']:,}"),
                         ("Posts",     f"{r[\'posts\']:,}"),
                         ("Private",   "Yes" if r[\'is_private\'] else "No"),
                         ("Verified",  "Yes" if r[\'is_verified\'] else "No")]:
            row = MDBoxLayout(size_hint_y=None, height=dp(24))
            row.add_widget(MDLabel(text=lbl, theme_text_color="Custom",
                                   text_color=get_color_from_hex("#475569"),
                                   font_style="Caption", size_hint_x=0.4))
            row.add_widget(MDLabel(text=val, theme_text_color="Custom",
                                   text_color=get_color_from_hex("#94a3b8"),
                                   font_style="Caption", bold=True, size_hint_x=0.6))
            box.add_widget(row)
    def bulk_scan(self):
        if self._scanning:
            self._stop = True; self.ids.bulk_btn.text = "START BULK SCAN"
            self._scanning = False; return
        path = self.ids.fpath.text.strip()
        if not os.path.exists(path):
            self.ids.plabel.text = "File not found."; return
        with open(path) as f:
            lines = [l.strip().lstrip("@") for l in f if l.strip()]
        self._stop = False; self._scanning = True
        self.ids.bulk_btn.text = "STOP"; self.ids.blog.text = ""
        threading.Thread(target=self._do_bulk, args=(lines,), daemon=True).start()
    def _do_bulk(self, us):
        db = Database(); total = len(us)
        def cb(cur, tot, r):
            line = f"@{r[\'username\']} → {r[\'status\']}\n"
            def upd(dt):
                self.ids.pbar.value   = cur/tot
                self.ids.plabel.text  = f"{cur}/{tot}"
                self.ids.blog.text   += line
            Clock.schedule_once(upd)
            db.save_scan(r)
        check_accounts_bulk(us, progress_cb=cb, stop_flag=lambda: self._stop)
        def done(dt):
            self._scanning = False; self.ids.bulk_btn.text = "START BULK SCAN"
            self.ids.plabel.text = "Done!"
        Clock.schedule_once(done)
''')

# ── screens/reports.py ──────────────────────────────────────
with open('/content/igapp/screens/reports.py','w') as f:
    f.write('''
import os, json
from datetime import datetime
from kivy.lang import Builder
from kivy.metrics import dp
from kivy.utils import get_color_from_hex
from kivymd.uix.screen import MDScreen
from kivymd.uix.boxlayout import MDBoxLayout
from kivymd.uix.label import MDLabel
from kivymd.toast import toast
from models.database import Database

Builder.load_string(\'\'\'<ReportsScreen>:
    name: "reports"
    MDBoxLayout:
        orientation: "vertical"
        md_bg_color: app.hex_rgba("#0a0a0f")
        MDBoxLayout:
            size_hint_y: None
            height: "60dp"
            padding: "16dp"
            md_bg_color: app.hex_rgba("#12121a")
            MDLabel:
                text: "[color=#22c55e]Reports[/color]"
                markup: True
                font_style: "H6"
                bold: True
        MDBoxLayout:
            size_hint_y: None
            height: "48dp"
            padding: ["10dp","4dp"]
            spacing: "8dp"
            md_bg_color: app.hex_rgba("#0a0a0f")
            MDRaisedButton:
                text: "Export HTML"
                md_bg_color: app.hex_rgba("#06b6d4")
                font_size: "12sp"
                on_release: root.export_html()
            MDRaisedButton:
                text: "Export JSON"
                md_bg_color: app.hex_rgba("#a855f7")
                font_size: "12sp"
                on_release: root.export_json()
            MDRaisedButton:
                text: "Clear All"
                md_bg_color: app.hex_rgba("#ef4444")
                font_size: "12sp"
                on_release: root.clear_all()
        MDLabel:
            id: count_lbl
            text: "Loading..."
            theme_text_color: "Custom"
            text_color: app.hex_rgba("#475569")
            font_style: "Caption"
            padding: ["16dp","4dp"]
            size_hint_y: None
            height: "22dp"
        ScrollView:
            MDBoxLayout:
                id: results_box
                orientation: "vertical"
                spacing: "2dp"
                size_hint_y: None
                height: self.minimum_height
\'\'\' )

SC = {"active":"#22c55e","disabled":"#ef4444","shadowban":"#f97316",
      "private":"#a855f7","unknown":"#94a3b8"}

def _dl():
    try:
        from android.storage import primary_external_storage_path
        return os.path.join(primary_external_storage_path(),"Download")
    except: return os.path.expanduser("~")

class ReportsScreen(MDScreen):
    def on_enter(self): self._load()
    def _load(self):
        scans = Database().get_all_scans()
        box = self.ids.results_box; box.clear_widgets()
        self.ids.count_lbl.text = f"{len(scans)} results"
        for s in scans:
            c = get_color_from_hex(SC.get(s["status"],"#94a3b8"))
            row = MDBoxLayout(size_hint_y=None, height=dp(34), padding=[dp(12),0], spacing=dp(4))
            row.add_widget(MDLabel(text=f"@{s[\'username\']}", theme_text_color="Custom",
                                   text_color=c, font_style="Body2", bold=True, size_hint_x=0.4))
            row.add_widget(MDLabel(text=s["status"].upper(), theme_text_color="Custom",
                                   text_color=c, font_style="Caption", size_hint_x=0.25))
            row.add_widget(MDLabel(text=f"{s[\'followers\']:,}", theme_text_color="Custom",
                                   text_color=get_color_from_hex("#94a3b8"),
                                   font_style="Caption", halign="right", size_hint_x=0.35))
            box.add_widget(row)
    def export_html(self):
        scans = Database().get_all_scans()
        rows = "".join(f"<tr><td>{s[\'username\']}</td><td>{s[\'status\']}</td>"
                       f"<td>{s[\'followers\']:,}</td></tr>" for s in scans)
        html = f"<html><body style=\'background:#0a0a0f;color:#f1f5f9\'>"\
               f"<h1>IG Manager Pro</h1><table border=1>{rows}</table></body></html>"
        path = os.path.join(_dl(),f"igmanager_{datetime.now().strftime(\'%Y%m%d_%H%M%S\')}.html")
        os.makedirs(_dl(),exist_ok=True)
        open(path,"w").write(html); toast(f"Saved: {path}")
    def export_json(self):
        scans = Database().get_all_scans()
        path = os.path.join(_dl(),f"igmanager_{datetime.now().strftime(\'%Y%m%d_%H%M%S\')}.json")
        os.makedirs(_dl(),exist_ok=True)
        json.dump(scans, open(path,"w"), ensure_ascii=False, indent=2)
        toast(f"Saved: {path}")
    def clear_all(self):
        Database().clear_all_scans(); self._load(); toast("All data cleared.")
''')

# ── screens/appeals.py ──────────────────────────────────────
with open('/content/igapp/screens/appeals.py','w') as f:
    f.write('''
import os
from datetime import datetime
from kivy.lang import Builder
from kivy.core.clipboard import Clipboard
from kivy.utils import get_color_from_hex
from kivymd.uix.screen import MDScreen
from kivymd.uix.menu import MDDropdownMenu
from kivymd.toast import toast
from models.database import Database
from core.appeals import generate_appeal, ISSUE_TYPES, INSTAGRAM_SUPPORT_URL

Builder.load_string(\'\'\'<AppealsScreen>:
    name: "appeals"
    MDBoxLayout:
        orientation: "vertical"
        md_bg_color: app.hex_rgba("#0a0a0f")
        MDBoxLayout:
            size_hint_y: None
            height: "60dp"
            padding: "16dp"
            md_bg_color: app.hex_rgba("#12121a")
            MDLabel:
                text: "[color=#ec4899]Support Hub[/color]"
                markup: True
                font_style: "H6"
                bold: True
        ScrollView:
            MDBoxLayout:
                orientation: "vertical"
                padding: "16dp"
                spacing: "12dp"
                size_hint_y: None
                height: self.minimum_height
                MDCard:
                    orientation: "vertical"
                    padding: "14dp"
                    spacing: "10dp"
                    radius: [14,]
                    md_bg_color: app.hex_rgba("#12121a")
                    size_hint_y: None
                    height: self.minimum_height
                    elevation: 4
                    MDTextField:
                        id: uname
                        hint_text: "Instagram Username"
                        mode: "rectangle"
                        size_hint_y: None
                        height: "46dp"
                    MDTextField:
                        id: fname
                        hint_text: "Full Legal Name"
                        mode: "rectangle"
                        size_hint_y: None
                        height: "46dp"
                    MDTextField:
                        id: email
                        hint_text: "Email"
                        mode: "rectangle"
                        size_hint_y: None
                        height: "46dp"
                    MDBoxLayout:
                        spacing: "10dp"
                        size_hint_y: None
                        height: "46dp"
                        MDTextField:
                            id: year
                            hint_text: "Year (e.g.2019)"
                            mode: "rectangle"
                            size_hint_x: 0.5
                        MDTextField:
                            id: country
                            hint_text: "Country"
                            mode: "rectangle"
                            size_hint_x: 0.5
                    MDRaisedButton:
                        id: issue_btn
                        text: "Select Issue Type"
                        md_bg_color: app.hex_rgba("#1a1a2e")
                        size_hint_x: 1
                        on_release: root.open_menu()
                    MDTextField:
                        id: detail
                        hint_text: "Describe the issue..."
                        mode: "rectangle"
                        multiline: True
                        size_hint_y: None
                        height: "100dp"
                    MDRaisedButton:
                        text: "GENERATE APPEAL"
                        md_bg_color: app.hex_rgba("#ec4899")
                        size_hint_x: 1
                        on_release: root.generate()
                MDCard:
                    orientation: "vertical"
                    padding: "14dp"
                    spacing: "10dp"
                    radius: [14,]
                    md_bg_color: app.hex_rgba("#12121a")
                    size_hint_y: None
                    height: self.minimum_height
                    elevation: 4
                    MDBoxLayout:
                        size_hint_y: None
                        height: "36dp"
                        spacing: "8dp"
                        MDRaisedButton:
                            text: "Arabic"
                            id: btn_ar
                            md_bg_color: app.hex_rgba("#a855f7")
                            font_size: "12sp"
                            on_release: root.show_tab(\'ar\')
                        MDRaisedButton:
                            text: "English"
                            id: btn_en
                            md_bg_color: app.hex_rgba("#1a1a2e")
                            font_size: "12sp"
                            on_release: root.show_tab(\'en\')
                    ScrollView:
                        size_hint_y: None
                        height: "220dp"
                        MDLabel:
                            id: out_text
                            text: "Fill the form above and tap Generate."
                            theme_text_color: "Custom"
                            text_color: app.hex_rgba("#94a3b8")
                            font_style: "Body2"
                            size_hint_y: None
                            height: self.texture_size[1]
                            text_size: self.width, None
                    MDBoxLayout:
                        size_hint_y: None
                        height: "42dp"
                        spacing: "8dp"
                        MDRaisedButton:
                            text: "Copy"
                            md_bg_color: app.hex_rgba("#1a1a2e")
                            font_size: "12sp"
                            on_release: root.copy_text()
                        MDRaisedButton:
                            text: "Save TXT"
                            md_bg_color: app.hex_rgba("#a855f7")
                            font_size: "12sp"
                            on_release: root.save_txt()
                        MDRaisedButton:
                            text: "IG Support"
                            md_bg_color: app.hex_rgba("#06b6d4")
                            font_size: "12sp"
                            on_release: root.open_support()
\'\'\' )

LABELS = {k: f"{v[0]} / {v[1]}" for k,v in ISSUE_TYPES.items()}

class AppealsScreen(MDScreen):
    _ar=""; _en=""; _tab="ar"; _key="disabled"; _menu=None
    def open_menu(self):
        items=[{"viewclass":"OneLineListItem","text":lbl,
                "on_release":lambda x=k: self._set(x)} for k,lbl in LABELS.items()]
        self._menu = MDDropdownMenu(caller=self.ids.issue_btn, items=items, width_mult=4)
        self._menu.open()
    def _set(self, k):
        self._key = k; self.ids.issue_btn.text = LABELS.get(k,k)
        if self._menu: self._menu.dismiss()
    def generate(self):
        data={"username":self.ids.uname.text.strip(),"full_name":self.ids.fname.text.strip(),
              "email":self.ids.email.text.strip(),"year_created":self.ids.year.text.strip(),
              "country":self.ids.country.text.strip(),"issue_type":self._key,
              "issue_detail":self.ids.detail.text.strip()}
        r = generate_appeal(data); self._ar=r["appeal_ar"]; self._en=r["appeal_en"]
        data.update(r); Database().save_appeal(data)
        self.show_tab(self._tab)
    def show_tab(self,t):
        self._tab=t
        self.ids.out_text.text = self._ar if t=="ar" else self._en
    def copy_text(self):
        Clipboard.copy(self.ids.out_text.text); toast("Copied!")
    def save_txt(self):
        try:
            from android.storage import primary_external_storage_path
            base=os.path.join(primary_external_storage_path(),"Download")
        except: base=os.path.expanduser("~")
        os.makedirs(base,exist_ok=True)
        tab="AR" if self._tab=="ar" else "EN"
        p=os.path.join(base,f"appeal_{tab}_{datetime.now().strftime(\'%Y%m%d_%H%M%S\')}.txt")
        open(p,"w",encoding="utf-8").write(self.ids.out_text.text); toast(f"Saved: {p}")
    def open_support(self):
        import webbrowser; webbrowser.open(INSTAGRAM_SUPPORT_URL)
''')

print('✅ All screen files created')

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 5: إنشاء main.py و buildozer.spec
# ═══════════════════════════════════════════════════════════
with open('/content/igapp/main.py','w',encoding='utf-8') as f:
    f.write('''
from kivy.config import Config
Config.set("graphics","width","412")
Config.set("graphics","height","892")

from kivy.lang import Builder
from kivy.utils import get_color_from_hex
from kivymd.app import MDApp
from kivymd.uix.navigationbar import MDNavigationBar, MDNavigationItem
from kivymd.uix.screenmanager import MDScreenManager
from kivymd.uix.screen import MDScreen
from screens.dashboard import DashboardScreen
from screens.scanner   import ScannerScreen
from screens.reports   import ReportsScreen
from screens.appeals   import AppealsScreen

KV = \'\'\'<Root>:
    orientation: "vertical"
    MDScreenManager:
        id: sm
    MDNavigationBar:
        id: nav
        on_switch_tabs: root.switch(*args)
        MDNavigationItem:
            icon: "view-dashboard"
            text: "Dashboard"
        MDNavigationItem:
            icon: "magnify"
            text: "Scanner"
        MDNavigationItem:
            icon: "chart-bar"
            text: "Reports"
        MDNavigationItem:
            icon: "shield-account"
            text: "Appeals"
\'\'\'
Builder.load_string(KV)

class Root(MDScreen):
    def switch(self, bar, item, icon, text):
        self.ids.sm.current = text.lower()

class IGManagerApp(MDApp):
    def build(self):
        self.theme_cls.primary_palette = "DeepPurple"
        self.theme_cls.accent_palette  = "Cyan"
        self.theme_cls.theme_style     = "Dark"
        root = Root()
        sm = root.ids.sm
        for Scr in [DashboardScreen,ScannerScreen,ReportsScreen,AppealsScreen]:
            sm.add_widget(Scr())
        sm.current = "dashboard"
        return root

    @staticmethod
    def hex_rgba(h, a=1.0):
        c = get_color_from_hex(h)
        return list(c[:3]) + [a]

    def on_start(self):
        try:
            from android.permissions import request_permissions, Permission
            request_permissions([Permission.READ_EXTERNAL_STORAGE,
                                  Permission.WRITE_EXTERNAL_STORAGE,
                                  Permission.INTERNET])
        except: pass

    def on_pause(self): return True
    def on_resume(self): pass

if __name__ == "__main__":
    IGManagerApp().run()
''')

with open('/content/igapp/buildozer.spec','w') as f:
    f.write("""[app]
title           = IG Manager Pro
package.name    = igmanagerpro
package.domain  = com.igmanager
source.dir      = .
source.include_exts = py,kv,png,jpg,ttf
version         = 1.0.0
requirements    = python3,kivy==2.3.0,kivymd==1.2.0,requests,certifi,urllib3,charset-normalizer,idna
orientation     = portrait
fullscreen      = 0

[buildozer]
log_level = 2

[android]
android.api             = 33
android.minapi          = 30
android.ndk             = 25c
android.arch            = arm64-v8a
android.permissions     = INTERNET,READ_EXTERNAL_STORAGE,WRITE_EXTERNAL_STORAGE
android.enable_androidx = True
p4a.bootstrap           = sdl2
""")

print('✅ main.py and buildozer.spec created')

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 6: بناء APK (البناء الأول يستغرق 25-40 دقيقة)
# ═══════════════════════════════════════════════════════════
import os, subprocess
os.chdir('/content/igapp')
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

print('🔨 Starting APK build... This takes 25-40 minutes on first run.')
print('☕ Go make some coffee and come back!')

result = subprocess.run(
    ['buildozer', 'android', 'debug'],
    capture_output=False,
    text=True
)

if result.returncode == 0:
    apks = [f for f in os.listdir('/content/igapp/bin') if f.endswith('.apk')]
    print(f'\n✅ SUCCESS! APK built: {apks}')
else:
    print(f'❌ Build failed. Check the log above.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 7: تحميل APK
# ═══════════════════════════════════════════════════════════
import os
from google.colab import files

bin_dir = '/content/igapp/bin'
apks = [f for f in os.listdir(bin_dir) if f.endswith('.apk')]

if apks:
    apk_path = os.path.join(bin_dir, apks[0])
    size_mb = os.path.getsize(apk_path) / (1024*1024)
    print(f'📱 APK: {apks[0]}')
    print(f'📦 Size: {size_mb:.1f} MB')
    print('⬇️  Downloading...')
    files.download(apk_path)
    print('✅ Download started!')
    print()
    print('📲 Installation steps:')
    print('1. Transfer APK to your Samsung S21 Ultra')
    print('2. Settings → Apps → Special Access → Install Unknown Apps')
    print('3. Enable for your file manager')
    print('4. Open APK from Downloads folder')
else:
    print('❌ No APK found. The build may have failed.')
    print('Check the output of Cell 6 for errors.')